# Sistema de Gestión de Proyectos de Ingeniería Civil – Villa María

**Trabajo práctico final – Fundamentos de Informática – Ingeniería Civil – UTN FRVM – 2026**

**Integrantes:** _completar_ &nbsp;|&nbsp; **Fecha de entrega:** 17/11/2026

## 1. Descripción del problema

La Municipalidad de Villa María inició un proyecto de renovación de la infraestructura urbana: un nuevo puente sobre el río Ctalamochita (barrio Palermo), la repavimentación de calles y el análisis de desagües pluviales. La Dirección de Obras Públicas necesita un programa en Python que permita:

1. Calcular **materiales y costo del puente** (hormigón y acero).
2. Analizar las **precipitaciones** y clasificar cada mes según el riesgo de inundación.
3. Planificar la **pavimentación** (superficie y volumen de asfalto).
4. **Dimensionar zapatas** según la carga y el tipo de suelo.
5. **Gestionar los ensayos de suelo**: profundidad mínima, humedad promedio e informe en archivo de texto.
6. **Controlar la calidad**: buscar proveedores según un criterio y analizar estadísticamente la resistencia.

## 2. Metodología

- Se usa **Python básico** (módulos `csv` y `math`): listas, diccionarios, funciones, bucles `for`/`while` y condicionales `if/elif/else`.
- Los archivos `.csv` se leen con una función propia, `leer_csv()`, que devuelve una lista de diccionarios.
- Cada parte tiene una celda de texto (qué se hace, fórmulas y supuestos) y una celda de código comentada.
- Los valores que son supuestos (umbrales, espesor, porcentajes) están en **constantes al inicio de cada parte** para poder modificarlos.
- Archivos necesarios en la misma carpeta que el notebook: `datos_climaticos_villa_maria.csv`, `calles_pavimentacion.csv`, `ensayos_suelo.csv` y `proveedores_materiales.csv`.

**Sobre el análisis de resistencia:** la consigna pide un "dataset de resultados de ensayos de probetas", pero el material entregado no incluye un archivo de probetas de hormigón. Por eso el análisis estadístico se hace sobre la columna `resistencia_kpa` de `ensayos_suelo.csv`, el único dato de resistencia disponible. Si se consiguiera un archivo de probetas, bastaría cambiar el archivo y la columna leídos en la sección 6.2.

In [1]:
# ---------------------------------------------------------------
# Preparación: módulos y funciones auxiliares
# ---------------------------------------------------------------
import csv      # leer archivos .csv
import math     # raíz cuadrada (sqrt) y redondeo hacia arriba (ceil)
import os       # comprobar que existen los archivos

def leer_csv(nombre_archivo):
    """Lee un CSV y devuelve una lista de diccionarios {columna: valor}.
    Todos los valores llegan como texto; después los convertimos a número."""
    if not os.path.exists(nombre_archivo):
        raise FileNotFoundError("No se encuentra " + nombre_archivo +
                                ". Ponelo en la misma carpeta que el notebook.")
    filas = []
    with open(nombre_archivo, encoding="utf-8-sig") as archivo:
        lector = csv.DictReader(archivo)    # cada fila queda como diccionario
        for fila in lector:
            filas.append(fila)
    return filas

def pedir_numero(mensaje):
    """Pide un número mayor que cero por teclado. Si el dato no es válido, vuelve a preguntar."""
    while True:
        texto = input(mensaje)
        try:
            numero = float(texto.replace(",", "."))   # acepta coma decimal
        except ValueError:
            print("  Eso no es un número. Probá de nuevo.")
            continue
        if numero <= 0:
            print("  Tiene que ser mayor que cero.")
            continue
        return numero

# Los ensayos de suelo se usan en varias partes, los leemos una sola vez
ensayos = leer_csv("ensayos_suelo.csv")
print("Ensayos de suelo leídos:", len(ensayos), "filas")

Ensayos de suelo leídos: 21 filas


---
# PARTE 1 – Cálculo de materiales del puente

**Qué se hace:** se piden las dimensiones con `input()`, se calcula el hormigón (losa + pilares), el acero de refuerzo y el costo estimado.

**Fórmulas:**
- Volumen de losa = largo × ancho × altura de losa
- Volumen de pilares = cantidad × lado a × lado b × altura del pilar
- Acero (kg) = 150 kg/m³ × volumen total de hormigón
- Costo hormigón = volumen × \$85.000/m³ &nbsp;|&nbsp; Costo acero = kg × \$1.200/kg

In [ ]:
# ---------- Constantes ----------
ACERO_POR_M3 = 150        # kg de acero por m³ de hormigón
PRECIO_HORMIGON = 85000   # $ por m³
PRECIO_ACERO = 1200       # $ por kg

# ---------- Entrada de datos ----------
print("=== DATOS DEL PUENTE ===")
largo = pedir_numero("Largo del puente (m): ")
ancho = pedir_numero("Ancho del puente (m): ")
altura_losa = pedir_numero("Altura (espesor) de la losa (m): ")
cant_pilares = int(pedir_numero("Cantidad de pilares: "))
lado_a = pedir_numero("Pilar - lado a (m): ")
lado_b = pedir_numero("Pilar - lado b (m): ")
altura_pilar = pedir_numero("Altura de cada pilar (m): ")

# ---------- Cálculos ----------
vol_losa = largo * ancho * altura_losa
vol_pilares = cant_pilares * lado_a * lado_b * altura_pilar
vol_hormigon = vol_losa + vol_pilares
acero_kg = vol_hormigon * ACERO_POR_M3
costo_hormigon = vol_hormigon * PRECIO_HORMIGON
costo_acero = acero_kg * PRECIO_ACERO
costo_total_puente = costo_hormigon + costo_acero

# ---------- Resultados ----------
print("\n=== RESULTADOS ===")
print(f"Volumen de losa:      {vol_losa:12,.2f} m3")
print(f"Volumen de pilares:   {vol_pilares:12,.2f} m3")
print(f"Hormigón total:       {vol_hormigon:12,.2f} m3")
print(f"Acero de refuerzo:    {acero_kg:12,.0f} kg")
print(f"Costo del hormigón:   $ {costo_hormigon:12,.0f}")
print(f"Costo del acero:      $ {costo_acero:12,.0f}")
print(f"COSTO TOTAL ESTIMADO: $ {costo_total_puente:12,.0f}")

---
# PARTE 2 – Análisis de precipitaciones

**Qué se hace:** se lee `datos_climaticos_villa_maria.csv` (22 meses: enero 2023 a octubre 2024). Como el archivo está ordenado por fecha, el **último año** son las **últimas 12 filas**. Se arma una lista con la precipitación mensual, se calculan promedio, máximo y mínimo, se identifican los meses sobre el promedio y se clasifica el riesgo de inundación con `if/elif/else`.

**Umbrales elegidos** (modificables; se definieron mirando los valores del último año):

| Precipitación mensual | Riesgo |
|---|---|
| menos de 50 mm | Bajo |
| de 50 mm a menos de 90 mm | Medio |
| 90 mm o más | Alto |

In [ ]:
datos_clima = leer_csv("datos_climaticos_villa_maria.csv")
ultimo_anio = datos_clima[-12:]      # las últimas 12 filas

# ---------- Listas con los datos del último año ----------
meses = []             # ej: "Noviembre 2023"
precipitaciones = []   # en mm
for fila in ultimo_anio:
    meses.append(fila["mes"] + " " + fila["año"])
    precipitaciones.append(float(fila["precipitacion_mm"]))

# ---------- Estadísticas ----------
promedio = sum(precipitaciones) / len(precipitaciones)
maximo = max(precipitaciones)
minimo = min(precipitaciones)
mes_max = meses[precipitaciones.index(maximo)]
mes_min = meses[precipitaciones.index(minimo)]
print(f"Promedio: {promedio:.1f} mm")
print(f"Máxima:   {maximo} mm ({mes_max})")
print(f"Mínima:   {minimo} mm ({mes_min})")

# ---------- Meses sobre el promedio ----------
meses_sobre_promedio = []
for i in range(len(precipitaciones)):
    if precipitaciones[i] > promedio:
        meses_sobre_promedio.append(meses[i])
print("\nMeses sobre el promedio (" + str(len(meses_sobre_promedio)) + "):", meses_sobre_promedio)

# ---------- Clasificación de riesgo ----------
UMBRAL_MEDIO = 50   # mm
UMBRAL_ALTO = 90    # mm

riesgos = []
print("\nMes               Precipitación   Riesgo")
for i in range(len(precipitaciones)):
    mm = precipitaciones[i]
    if mm >= UMBRAL_ALTO:
        riesgo = "ALTO"
    elif mm >= UMBRAL_MEDIO:
        riesgo = "MEDIO"
    else:
        riesgo = "BAJO"
    riesgos.append(riesgo)
    print(f"{meses[i]:<16} {mm:8.1f} mm      {riesgo}")

n_bajo = riesgos.count("BAJO")
n_medio = riesgos.count("MEDIO")
n_alto = riesgos.count("ALTO")
print(f"\nResumen: {n_bajo} meses de riesgo bajo, {n_medio} medio y {n_alto} alto")

In [ ]:
# ---------- Gráfico (opcional) ----------
import matplotlib.pyplot as plt

colores = {"BAJO": "green", "MEDIO": "orange", "ALTO": "red"}
plt.figure(figsize=(10, 4))
plt.bar(meses, precipitaciones, color=[colores[r] for r in riesgos])
plt.axhline(promedio, linestyle="--", color="black", label="Promedio")
plt.xticks(rotation=60)
plt.ylabel("Precipitación (mm)")
plt.title("Precipitación mensual y riesgo de inundación")
plt.legend()
plt.tight_layout()
plt.show()

---
# PARTE 3 – Planificación de pavimentación

**Qué se hace:** se lee `calles_pavimentacion.csv` y se arma un **diccionario** `{calle: longitud}`. La superficie de cada calle es **longitud × ancho** (el ancho se guarda en otro diccionario). El asfalto necesario es **superficie × espesor (0,05 m)**.

In [ ]:
ESPESOR_ASFALTO = 0.05   # m (5 cm)

calles = leer_csv("calles_pavimentacion.csv")

longitudes = {}   # {calle: longitud en m}
anchos = {}       # {calle: ancho en m}
for fila in calles:
    nombre = fila["nombre_calle"]
    longitudes[nombre] = float(fila["longitud_metros"])
    anchos[nombre] = float(fila["ancho_metros"])

print("Calles a pavimentar y su longitud:")
for nombre in longitudes:
    print(f"  {nombre:<18} {longitudes[nombre]:6.0f} m")

# Acumuladores: arrancan en cero y se suma calle por calle
longitud_total = 0
superficie_total = 0
for nombre in longitudes:
    longitud_total = longitud_total + longitudes[nombre]
    superficie_total = superficie_total + longitudes[nombre] * anchos[nombre]

volumen_asfalto = superficie_total * ESPESOR_ASFALTO

print(f"\nCantidad de calles:  {len(longitudes)}")
print(f"Longitud total:      {longitud_total:,.0f} m")
print(f"Superficie total:    {superficie_total:,.0f} m2")
print(f"Asfalto necesario:   {volumen_asfalto:,.1f} m3 (espesor {ESPESOR_ASFALTO*100:.0f} cm)")

---
# PARTE 4 – Función de cálculo estructural: `dimensionar_zapata()`

**Qué se hace:** una función que dimensiona una zapata aislada usando los ensayos de `ensayos_suelo.csv`.

**Método:**
1. Se busca, entre los ensayos del `punto_muestreo`, el de profundidad **más cercana** a `profundidad_deseada`.
2. La **resistencia** de ese ensayo (kPa) se toma como tensión admisible del suelo.
3. La carga se aumenta un **10 %** por el peso propio de la zapata (hipótesis simplificada).
4. **Área necesaria** = carga mayorada / tensión admisible (kN / kPa = m²).
5. Lados: *cuadrada* → lado = √área; *rectangular* → ancho = √(área / relación) y largo = relación × ancho.
6. Las medidas se **redondean hacia arriba** a múltiplos de 5 cm (si se redondeara hacia abajo, el suelo trabajaría por encima de su capacidad).
7. Se verifica la tensión de contacto real: carga mayorada / (B × L) ≤ tensión admisible.

La función devuelve `None` y avisa si hay un dato inválido (punto inexistente, forma incorrecta, carga negativa).

In [ ]:
PESO_PROPIO = 0.10   # 10 % de la carga

def redondear_5cm(medida):
    """Redondea hacia arriba a múltiplos de 5 cm (1 m = 20 pasos de 5 cm)."""
    return math.ceil(round(medida * 20, 6)) / 20

def dimensionar_zapata(carga_columna, punto_muestreo, profundidad_deseada, forma, relacion_lados):
    """Dimensiona una zapata aislada.
    carga_columna: kN | punto_muestreo: 'P1'..'P7' | profundidad_deseada: m
    forma: 'cuadrada' o 'rectangular' | relacion_lados: largo/ancho (1.0 si es cuadrada)
    Devuelve un diccionario con los resultados, o None si hay un error."""
    # ---------- Validaciones ----------
    if carga_columna <= 0:
        print("Error: la carga debe ser mayor que cero.")
        return None
    if forma != "cuadrada" and forma != "rectangular":
        print("Error: la forma debe ser 'cuadrada' o 'rectangular'.")
        return None
    if forma == "rectangular" and relacion_lados <= 0:
        print("Error: la relación de lados debe ser mayor que cero.")
        return None
    if forma == "cuadrada":
        relacion_lados = 1.0

    # ---------- Ensayo del punto con profundidad más cercana ----------
    mejor_ensayo = None
    menor_diferencia = None
    for fila in ensayos:
        if fila["punto_muestreo"] == punto_muestreo:
            diferencia = abs(float(fila["profundidad_m"]) - profundidad_deseada)
            if menor_diferencia is None or diferencia < menor_diferencia:
                menor_diferencia = diferencia
                mejor_ensayo = fila
    if mejor_ensayo is None:
        print("Error: el punto", punto_muestreo, "no existe en ensayos_suelo.csv.")
        return None

    tension_adm = float(mejor_ensayo["resistencia_kpa"])   # kPa = kN/m2

    # ---------- Cálculo ----------
    carga_total = carga_columna * (1 + PESO_PROPIO)
    area = carga_total / tension_adm
    ancho = math.sqrt(area / relacion_lados)
    largo = relacion_lados * ancho
    ancho = redondear_5cm(ancho)
    largo = redondear_5cm(largo)
    tension_real = carga_total / (ancho * largo)

    return {"punto": punto_muestreo,
            "tipo_suelo": mejor_ensayo["tipo_suelo"],
            "profundidad_usada": float(mejor_ensayo["profundidad_m"]),
            "tension_admisible": tension_adm,
            "carga_total": carga_total,
            "forma": forma,
            "ancho": ancho, "largo": largo,
            "area": ancho * largo,
            "tension_real": tension_real}

def mostrar_zapata(r):
    """Imprime el resultado de dimensionar_zapata() de forma ordenada."""
    if r is None:
        print("  (sin resultado)\n")
        return
    print(f"  Suelo: {r['tipo_suelo']} (ensayo a {r['profundidad_usada']} m) - tensión admisible {r['tension_admisible']:.0f} kPa")
    print(f"  Carga mayorada: {r['carga_total']:.1f} kN")
    print(f"  Zapata {r['forma']}: {r['ancho']:.2f} m x {r['largo']:.2f} m  (área {r['area']:.2f} m2)")
    print(f"  Tensión de contacto: {r['tension_real']:.1f} kPa  ->",
          "OK" if r["tension_real"] <= r["tension_admisible"] else "NO CUMPLE")
    print()

In [ ]:
print("Ejemplo 1: cuadrada, 600 kN, P2 a 3 m")
mostrar_zapata(dimensionar_zapata(600, "P2", 3.0, "cuadrada", 1.0))

print("Ejemplo 2: rectangular (relación 1.5), 900 kN, P4 a 3 m")
mostrar_zapata(dimensionar_zapata(900, "P4", 3.0, "rectangular", 1.5))

print("Ejemplo 3: profundidad sin ensayo exacto (2 m en P1 -> usa el más cercano)")
mostrar_zapata(dimensionar_zapata(400, "P1", 2.0, "cuadrada", 1.0))

print("Ejemplo 4: control de errores (punto P9 inexistente)")
mostrar_zapata(dimensionar_zapata(300, "P9", 1.5, "cuadrada", 1.0))

---
# PARTE 5 – Gestión de archivos de proyecto

**Qué se hace:**
- **5.1** El usuario ingresa un punto de muestreo y una resistencia; el programa informa la **profundidad mínima** que la alcanza.
- **5.2** Se calcula la **humedad promedio** de cada punto.
- **5.3** Se guardan los resultados en un **archivo de texto** con formato de informe.

Los ensayos ya se leyeron al principio (variable `ensayos`).

### 5.1 Profundidad mínima para una resistencia dada
Se arma la lista de capas `[profundidad, resistencia]` del punto, se ordena de menor a mayor profundidad y se devuelve la primera capa cuya resistencia sea mayor o igual a la pedida.

In [ ]:
def profundidad_minima(punto, resistencia_pedida):
    """Menor profundidad (m) del punto donde la resistencia alcanza el valor pedido.
    Devuelve None si ninguna profundidad ensayada llega a ese valor."""
    capas = []
    for fila in ensayos:
        if fila["punto_muestreo"] == punto:
            capas.append([float(fila["profundidad_m"]), float(fila["resistencia_kpa"])])
    capas.sort()                     # de menor a mayor profundidad
    for capa in capas:
        if capa[1] >= resistencia_pedida:
            return capa[0]
    return None

# ---------- Programa con input ----------
puntos = []                          # puntos distintos que hay en el archivo
for fila in ensayos:
    if fila["punto_muestreo"] not in puntos:
        puntos.append(fila["punto_muestreo"])
print("Puntos de muestreo disponibles:", puntos)

punto_consulta = input("Punto de muestreo: ").strip().upper()
while punto_consulta not in puntos:
    print("  Ese punto no existe.")
    punto_consulta = input("Punto de muestreo: ").strip().upper()
resistencia_consulta = pedir_numero("Resistencia requerida (kPa): ")

prof_consulta = profundidad_minima(punto_consulta, resistencia_consulta)
if prof_consulta is None:
    print("En", punto_consulta, "ninguna profundidad ensayada alcanza", resistencia_consulta, "kPa.")
else:
    print(f"En {punto_consulta}, la resistencia de {resistencia_consulta:.0f} kPa se alcanza desde los {prof_consulta} m de profundidad.")

Puntos de muestreo disponibles: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7']


### 5.2 Humedad promedio por punto de muestreo
Un diccionario guarda, para cada punto, la **lista** de humedades medidas; después se promedia cada lista.

In [ ]:
humedades = {}    # {punto: [humedad1, humedad2, ...]}
for fila in ensayos:
    punto = fila["punto_muestreo"]
    if punto not in humedades:
        humedades[punto] = []
    humedades[punto].append(float(fila["humedad_porcentaje"]))

humedad_promedio = {}   # {punto: promedio}
for punto in humedades:
    humedad_promedio[punto] = sum(humedades[punto]) / len(humedades[punto])

print("Humedad promedio por punto de muestreo:")
for punto in humedad_promedio:
    print(f"  {punto}: {humedad_promedio[punto]:.2f} %")

Humedad promedio por punto de muestreo:
  P1: 16.50 %
  P2: 19.77 %
  P3: 10.60 %
  P4: 18.50 %
  P5: 21.87 %
  P6: 8.17 %
  P7: 23.13 %


### 5.3 Guardar el informe en un archivo de texto
Se abre el archivo con `open(..., "w")` (modo escritura) y se escribe línea por línea con `write()`. En Colab el archivo aparece en el panel de archivos de la izquierda.

In [ ]:
nombre_informe = "informe_ensayos_suelo.txt"

with open(nombre_informe, "w", encoding="utf-8") as informe:
    informe.write("=" * 62 + "\n")
    informe.write("INFORME DE ENSAYOS DE SUELO - VILLA MARIA\n")
    informe.write("=" * 62 + "\n\n")
    informe.write("Archivo de origen: ensayos_suelo.csv\n")
    informe.write("Cantidad de ensayos: " + str(len(ensayos)) + "\n")
    informe.write("Puntos de muestreo: " + str(len(puntos)) + "\n\n")

    informe.write("1) HUMEDAD PROMEDIO POR PUNTO\n")
    informe.write("-" * 62 + "\n")
    for punto in humedad_promedio:
        informe.write(f"  {punto}: {humedad_promedio[punto]:.2f} %\n")

    informe.write("\n2) RESISTENCIA POR PUNTO Y PROFUNDIDAD\n")
    informe.write("-" * 62 + "\n")
    informe.write(f"  {'Punto':<7}{'Prof. (m)':>10}{'Resist. (kPa)':>15}  Suelo\n")
    for fila in ensayos:
        informe.write(f"  {fila['punto_muestreo']:<7}{float(fila['profundidad_m']):>10.1f}"
                      f"{float(fila['resistencia_kpa']):>15.0f}  {fila['tipo_suelo']}\n")

    informe.write("\n3) CONSULTA REALIZADA\n")
    informe.write("-" * 62 + "\n")
    informe.write(f"  Punto: {punto_consulta} | Resistencia requerida: {resistencia_consulta:.0f} kPa\n")
    if prof_consulta is None:
        informe.write("  Resultado: ninguna profundidad ensayada alcanza ese valor.\n")
    else:
        informe.write(f"  Resultado: profundidad minima = {prof_consulta} m\n")

print("Informe guardado en:", nombre_informe)
print()
with open(nombre_informe, encoding="utf-8") as informe:   # lo mostramos para verificar
    print(informe.read())

Informe guardado en: informe_ensayos_suelo.txt

INFORME DE ENSAYOS DE SUELO - VILLA MARIA

Archivo de origen: ensayos_suelo.csv
Cantidad de ensayos: 21
Puntos de muestreo: 7

1) HUMEDAD PROMEDIO POR PUNTO
--------------------------------------------------------------
  P1: 16.50 %
  P2: 19.77 %
  P3: 10.60 %
  P4: 18.50 %
  P5: 21.87 %
  P6: 8.17 %
  P7: 23.13 %

2) RESISTENCIA POR PUNTO Y PROFUNDIDAD
--------------------------------------------------------------
  Punto   Prof. (m)  Resist. (kPa)  Suelo
  P1            1.5            120  Arcilla
  P1            3.0            145  Arcilla
  P1            5.0            180  Limo arcilloso
  P2            1.5             95  Limo
  P2            3.0            135  Arcilla limosa
  P2            5.0            165  Arcilla
  P3            1.5            110  Arena limosa
  P3            3.0            125  Arena
  P3            5.0            220  Grava arenosa
  P4            1.5            115  Arcilla
  P4            3.0           

---
# PARTE 6 – Sistema de control de calidad

## 6.1 Búsqueda de proveedores

**Qué se hace:** `buscar_proveedores(material, prioridad)` busca en `proveedores_materiales.csv` los proveedores del material pedido, los **ordena** según la prioridad y muestra su teléfono. La búsqueda ignora mayúsculas y tildes, y acepta nombres parciales (`acero` encuentra todos los aceros).

| Prioridad | Criterio de orden |
|---|---|
| `"calidad"` | de mayor a menor calidad (más es mejor) |
| `"precio"` | de menor a mayor precio unitario (menos es mejor) |
| `"distancia"` | de menor a mayor distancia (más cerca es mejor) |

In [ ]:
def sin_tildes(texto):
    """Pasa a minúscula y saca tildes, para comparar 'Hormigón' con 'hormigon'."""
    texto = texto.lower()
    cambios = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u"}
    for con_tilde in cambios:
        texto = texto.replace(con_tilde, cambios[con_tilde])
    return texto

def buscar_proveedores(material, prioridad):
    """Imprime los proveedores del material ordenados según la prioridad.
    prioridad: 'calidad', 'precio' o 'distancia'."""
    prioridad = prioridad.strip().lower()
    if prioridad not in ("calidad", "precio", "distancia"):
        print("Error: la prioridad debe ser 'calidad', 'precio' o 'distancia'.")
        return

    # Proveedores cuyo material contiene el texto buscado
    encontrados = []
    for fila in leer_csv("proveedores_materiales.csv"):
        if sin_tildes(material) in sin_tildes(fila["material"]):
            encontrados.append({"proveedor": fila["proveedor"],
                                "material": fila["material"],
                                "telefono": fila["telefono"],
                                "calidad": float(fila["calidad"]),
                                "precio": float(fila["precio_unitario"]),
                                "distancia": float(fila["distancia_km"])})
    if len(encontrados) == 0:
        print("No se encontraron proveedores para '" + material + "'.")
        return

    # Ordenar según la prioridad (key = campo por el que se ordena)
    if prioridad == "calidad":
        encontrados.sort(key=lambda p: p["calidad"], reverse=True)   # mayor primero
    elif prioridad == "precio":
        encontrados.sort(key=lambda p: p["precio"])                  # menor primero
    else:
        encontrados.sort(key=lambda p: p["distancia"])               # menor primero

    print(f"Proveedores de '{material}' - prioridad: {prioridad}")
    print(f"{'Proveedor':<26}{'Material':<18}{'Teléfono':<14}{'Calidad':>8}{'Precio':>10}{'Dist.(km)':>10}")
    for p in encontrados:
        print(f"{p['proveedor']:<26}{p['material']:<18}{p['telefono']:<14}"
              f"{p['calidad']:>8.1f}{p['precio']:>10,.0f}{p['distancia']:>10.1f}")
    print()

# ---------- Ejemplos ----------
buscar_proveedores("hormigón h25", "precio")
buscar_proveedores("acero", "calidad")

# ---------- Programa con input ----------
material_pedido = input("Material a buscar: ")
prioridad_pedida = input("Prioridad (calidad / precio / distancia): ")
buscar_proveedores(material_pedido, prioridad_pedida)

Proveedores de 'hormigón h25' - prioridad: precio
Proveedor                 Material          Teléfono       Calidad    Precio Dist.(km)
Materiales del Sur        Hormigón H25      353-4534567        8.8    83,000      12.4
Hormigones VM SA          Hormigón H25      353-4521000        9.2    85,000       5.3
La Casa del Constructor   Hormigón H25      353-4545678        8.9    86,000       4.8

Proveedores de 'acero' - prioridad: calidad
Proveedor                 Material          Teléfono       Calidad    Precio Dist.(km)
Distribuidora Córdoba     Acero ADN500      351-4123456        9.4     1,290      45.2
Aceros Centro SRL         Acero ADN500      353-4567890        9.3     1,350       8.7
Hierros del Centro        Acero ADN500      353-4590123        9.1     1,320       3.5
Aceros Centro SRL         Acero ADN420      353-4567890        9.0     1,200       8.7
Hierros del Centro        Acero ADN420      353-4590123        8.9     1,180       3.5

Material a buscar: acero
Prioridad

## 6.2 Análisis estadístico de la resistencia

**Qué se hace:** se analiza la columna `resistencia_kpa` de `ensayos_suelo.csv` (ver la nota de la sección 2). Se calculan:

- **Media** $\bar{x} = \dfrac{\sum x_i}{n}$
- **Mediana**, **mínimo**, **máximo** y **rango**
- **Desvío estándar muestral** $s = \sqrt{\dfrac{\sum (x_i - \bar{x})^2}{n-1}}$
- **Coeficiente de variación** $CV = \dfrac{s}{\bar{x}} \cdot 100$ (%)
- **Resistencia característica** $R_k \approx \bar{x} - 1{,}645 \cdot s$ (valor que supera el 95 % de los ensayos)
- Cantidad de ensayos por debajo de la resistencia mínima requerida (**100 kPa**, modificable)

In [ ]:
RESISTENCIA_MINIMA = 100   # kPa (valor requerido, modificable)

resistencias = []
for fila in ensayos:
    resistencias.append(float(fila["resistencia_kpa"]))
n = len(resistencias)

# Media
media = sum(resistencias) / n

# Desvío estándar muestral (se divide por n - 1)
suma_cuadrados = 0
for r in resistencias:
    suma_cuadrados = suma_cuadrados + (r - media) ** 2
desvio = math.sqrt(suma_cuadrados / (n - 1))

# Otros estadísticos
minimo_r = min(resistencias)
maximo_r = max(resistencias)
rango = maximo_r - minimo_r
cv = desvio / media * 100

ordenadas = sorted(resistencias)
if n % 2 == 1:
    mediana = ordenadas[n // 2]
else:
    mediana = (ordenadas[n // 2 - 1] + ordenadas[n // 2]) / 2

resistencia_caracteristica = media - 1.645 * desvio

bajo_minimo = 0
for r in resistencias:
    if r < RESISTENCIA_MINIMA:
        bajo_minimo = bajo_minimo + 1

print(f"Cantidad de ensayos:       {n}")
print(f"Media:                     {media:.1f} kPa")
print(f"Mediana:                   {mediana:.1f} kPa")
print(f"Mínimo / Máximo / Rango:   {minimo_r:.0f} / {maximo_r:.0f} / {rango:.0f} kPa")
print(f"Desvío estándar:           {desvio:.1f} kPa")
print(f"Coeficiente de variación:  {cv:.1f} %")
print(f"Resistencia característica: {resistencia_caracteristica:.1f} kPa")
print(f"Ensayos bajo {RESISTENCIA_MINIMA} kPa:       {bajo_minimo} de {n} ({bajo_minimo / n * 100:.1f} %)")
if resistencia_caracteristica >= RESISTENCIA_MINIMA:
    print("-> La resistencia característica cumple el mínimo requerido.")
else:
    print("-> La resistencia característica NO alcanza el mínimo requerido.")

Cantidad de ensayos:       21
Media:                     143.6 kPa
Mediana:                   135.0 kPa
Mínimo / Máximo / Rango:   75 / 245 / 170 kPa
Desvío estándar:           43.2 kPa
Coeficiente de variación:  30.1 %
Resistencia característica: 72.5 kPa
Ensayos bajo 100 kPa:       3 de 21 (14.3 %)
-> La resistencia característica NO alcanza el mínimo requerido.


In [ ]:
# ---------- Histograma (opcional) ----------
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.hist(resistencias, bins=8, edgecolor="black")
plt.axvline(media, color="blue", label="Media")
plt.axvline(RESISTENCIA_MINIMA, color="red", linestyle="--", label="Mínimo requerido")
plt.xlabel("Resistencia (kPa)")
plt.ylabel("Cantidad de ensayos")
plt.title("Distribución de la resistencia del suelo")
plt.legend()
plt.show()

---
# 3. Resultados obtenidos y conclusiones

Los datos del puente corresponden a una corrida de ejemplo (60 m × 9 m, losa de 0,30 m, 4 pilares de 1,0 × 1,2 m y 6 m de altura; consulta de suelo en P5 con 150 kPa; proveedores de acero por distancia). Si se ingresan otros datos, los resultados cambian.

| Parte | Resultado principal |
|---|---|
| Puente | 190.80 m³ de hormigón (162.0 m³ de losa + 28.8 m³ de pilares), 28,620 kg de acero y un costo total estimado de \$50,562,000 |
| Precipitaciones | Promedio de 60.5 mm; máximo 112.5 mm (Noviembre 2023); mínimo 10.5 mm (Junio 2024). 6 meses superan el promedio. Riesgo: 5 meses bajo, 4 medio y 3 alto |
| Pavimentación | 20 calles, 8,445 m de longitud, 57,660 m² de superficie y 2,883.0 m³ de asfalto (espesor 5 cm) |
| Zapatas | Se dimensionaron con la tensión admisible del ensayo más cercano a la profundidad pedida; las medidas se redondean hacia arriba a 5 cm y se verifica la tensión de contacto |
| Suelos | Humedad promedio entre 8.2 % y 23.1 %; consulta en P5: 150 kPa se alcanza desde 5.0 m; informe guardado en `informe_ensayos_suelo.txt` |
| Proveedores | Búsqueda por calidad, precio o distancia con nombre y teléfono |
| Estadística | Media 143.6 kPa, desvío 43.2 kPa, CV 30.1 %, resistencia característica 72.5 kPa; 3 de 21 ensayos bajo 100 kPa |

## Conclusiones

- El programa automatiza los cálculos de materiales, pavimentación, dimensionado de zapatas y control de calidad que la Dirección de Obras Públicas necesita, y valida los datos ingresados por el usuario.
- **Precipitaciones:** el riesgo alto se concentra en los meses de verano, lo que debe tenerse en cuenta para el diseño de los desagües pluviales y la planificación de las obras.
- **Limitaciones:** la tensión admisible se tomó directamente de la resistencia del ensayo y el peso propio de la zapata se estimó en 10 %; en un proyecto real se usarían las verificaciones de la normativa (capacidad portante, asentamientos, factor de seguridad). Los umbrales de riesgo de inundación son criterios propios del grupo.
- **Análisis de resistencia:** se hizo sobre los ensayos de suelo porque no se contó con un archivo de probetas de hormigón; con ese archivo bastaría cambiar el origen de datos.